### Import libraries

In [2]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

### Load the dataset

In [4]:
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

df = pd.read_csv(
    ROOT / "data/processed/model_ready_paddy_qa.csv",
    keep_default_na=False,
    encoding="utf-8-sig",
)

### split dataset

In [6]:
issues = df[["issue_id", "label"]].drop_duplicates()

assert issues["issue_id"].is_unique

assert issues["label"].value_counts().min() >= 3, (
    "A category has too few issue groups for stratified splitting."
)

trainval_issues, test_issues = train_test_split(
    issues,
    test_size=0.15,
    random_state=42,
    stratify=issues["label"],
)

train_issues, val_issues = train_test_split(
    trainval_issues,
    test_size=16 / 90,
    random_state=42,
    stratify=trainval_issues["label"],
)

split_ids = {
    "train": set(train_issues["issue_id"]),
    "validation": set(val_issues["issue_id"]),
    "test": set(test_issues["issue_id"]),
}

assert not (split_ids["train"] & split_ids["validation"])
assert not (split_ids["train"] & split_ids["test"])
assert not (split_ids["validation"] & split_ids["test"])

for name, issue_ids in split_ids.items():
    part = df[df["issue_id"].isin(issue_ids)].copy()

    output = ROOT / f"data/processed/paddy_{name}.csv"

    part.to_csv(
        output,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        name,
        "records:", len(part),
        "issue groups:", part["issue_id"].nunique(),
    )

train records: 17760 issue groups: 74
validation records: 3840 issue groups: 16
test records: 3840 issue groups: 16
